# 운영 모니터링 대시보드 — 소스 확정 & PING 정의 (작업 노트)

**목적**: 일별 서비스 지표를 모니터링하며 급락/이상 징후를 빠르게 잡는 운영 대시보드.
일회성 분석(votes 리텐션)과는 **별개 트랙**이며, 이 노트북은 그 대시보드의 **데이터 소스와 지표 정의를 확정하는 과정**을 정리한다.

---

## ✅ 결정된 것
| 항목 | 결정 | 근거 |
|---|---|---|
| **소스** | `log_Data.deep_data` (hackle 13개월 일별 로그) | 일별 연속 + user_id 보유 + 커버리지 넓음 |
| **PING 발신** | `complete_question_question` (질문 답하며 지목 = 발신) | 팀 확인 완료 |
| **user_id** | STRING 원본 유지, **`SAFE_CAST` 금지** | hash형 유저 63%가 캐스팅에서 증발 |
| **결제 / 탈퇴 / 가입** | votes `stg_payment` / `stg_withdraw` / `stg_users` | deep_data엔 결제 '완료' 이벤트 없음 |

## ⚠️ 팀 논의 필요
1. **DAU 명칭·정의**: `launch_app` distinct user_id 로 확정하되, hash/numeric 혼재로 **동일인 이중계상 가능성**을 어떻게 표기할지
2. **힌트 소스**: 운영용 deep_data `click_question_hint`(9/13~) vs 활성화 정의 정합용 votes `opened_times>0`(pointhistory) — 어느 쪽을 쓸지
3. **힌트 두 갈래**: `click_question_hint` 와 `click_question_hint_initial`(초성) 을 합칠지 분리할지
4. **공통 유효창 시작일**: PING/DAU는 6/20~, 힌트는 9/13~ → 대시보드 시작을 언제로 통일할지
5. **리텐션 패널**: votes 분석 결과(정적)를 운영 대시보드에 넣을지, 뺄지


## 0. 환경 설정

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
%load_ext google.cloud.bigquery

## 1. 소스 선정 — 왜 `deep_data`인가

운영 대시보드는 **① 일별 연속 + ② 긴 기간 + ③ 유저 식별**이 동시에 필요하다.

| 소스 | 일별 연속 | 기간 | user_id | 판정 |
|---|---|---|---|---|
| votes(stg_vote) | ✓ | 연속 | ✓ | 발신자 4,849명뿐 → 커버리지 협소 |
| event_log | ✓ | 25일 | ✗(session만) | 너무 짧음 |
| **deep_data** | **✓** | **13개월** | **✓** | **✅ 채택** |

`deep_data`는 6일 간격 JSON 파일이지만 **파일당 6일치가 담겨** 있어, 이어붙이면 일별 연속이 된다. (파일명만 보고 '6일 샘플'로 오판했다가 아래 쿼리로 정정)

In [ ]:
%%bigquery --project your-gcp-project
-- deep_data가 '일별 연속'인지 확인 → 348일 연속(2023-04-07 ~ 2024-04-30)으로 확인됨
SELECT
  DATE(local_timestamp)   AS evt_date,
  COUNT(*)                AS row_cnt,
  COUNT(DISTINCT user_id) AS users
FROM `your-gcp-project.log_Data.deep_data`
GROUP BY evt_date
ORDER BY evt_date;

## 2. 데이터 특성 & 이벤트 파악

**월별 개요** — 결과 요약:
- `2023-04~05`: row는 폭발적(3,800만+)이나 **user_id 전부 0, event_key 3종뿐** → 유저 식별 불가, **배제**
- `2023-06~2024-04`: user_id 채워지기 시작 + event_key 28~76종 → 분석 가능 구간
- **중복 없음**: 모든 월에서 `row_cnt == distinct_event_ids`

In [ ]:
%%bigquery --project your-gcp-project
-- 월별 개요: 중복(row vs distinct id) / user_id 결측 / event_key 다양성
SELECT
  FORMAT_DATE('%Y-%m', DATE(local_timestamp)) AS ym,
  COUNT(*)                                 AS row_cnt,
  COUNT(DISTINCT id)                        AS distinct_event_ids,
  COUNT(DISTINCT user_id)                   AS users,
  COUNTIF(user_id IS NULL OR user_id = '')  AS null_user,
  COUNT(DISTINCT event_key)                 AS event_keys
FROM `your-gcp-project.log_Data.deep_data`
GROUP BY ym ORDER BY ym;

**event_key 목록** — 운영 지표 매핑:
- **DAU** = `launch_app` (그날 앱 켠 유저)
- **PING 발신** = `complete_question_question` (질문 답하며 지목)
- **힌트** = `click_question_hint` (+ `click_question_hint_initial` 초성)
- **결제** = `click_purchase`(시도)만 있고 **완료 이벤트 없음** → 결제는 votes `stg_payment` 사용

In [ ]:
%%bigquery --project your-gcp-project
-- event_key 실제 목록 + 유저 커버리지 (유효 구간 기준)
SELECT
  event_key,
  COUNT(*)                AS row_cnt,
  COUNT(DISTINCT user_id) AS users
FROM `your-gcp-project.log_Data.deep_data`
WHERE DATE(local_timestamp) >= '2023-06-01'
  AND user_id IS NOT NULL AND user_id != ''
GROUP BY event_key
ORDER BY row_cnt DESC
LIMIT 60;

## 3. ⭐ user_id 함정 (가장 중요한 검증)

두 가지가 지표를 오염시킬 수 있어 반드시 확인했다.

### 3-1. 이벤트마다 user_id 채워지는 시점이 다르다
- `launch_app`, `complete_question_question`: **2023-06-20~**
- `click_question_hint`(+initial): **2023-09-13~** (이벤트 자체가 늦게 시작)

→ 대시보드에서 힌트 라인은 9/13부터만 유효. 지표별 시작일이 다름을 명시해야 함.

In [ ]:
%%bigquery --project your-gcp-project
-- 핵심 이벤트별 user_id가 실제로 채워지기 시작하는 날
SELECT
  event_key,
  MIN(DATE(local_timestamp)) AS first_any,
  MIN(IF(user_id IS NOT NULL AND user_id!='', DATE(local_timestamp), NULL)) AS first_with_user,
  MAX(DATE(local_timestamp)) AS last_date
FROM `your-gcp-project.log_Data.deep_data`
WHERE event_key IN ('launch_app','$session_start','complete_question_question',
                    'click_question_hint','click_question_hint_initial')
GROUP BY event_key ORDER BY event_key;

### 3-2. user_id 형식이 3종 혼재 → `SAFE_CAST` 쓰면 안 됨

`complete_question_question`의 user_id 형식 분류 결과:

| 형식 | row | distinct |
|---|---|---|
| **hash** (Firebase UID형, `001zjRPEG...`) | 218,733 | 72,690 |
| **numeric** (votes id형, `1000006`) | 126,180 | 42,505 |
| empty | 27,741 | 0 |

**`SAFE_CAST(user_id AS INT64)`를 쓰면 hash형 72,690명(=발신 row의 63%)이 전부 NULL로 죽는다.**
6~7월은 hash형이라 통째로 증발 → 이게 초기 `ping_users=0` 버그의 원인이었다.

→ **결론: user_id는 STRING 원본 그대로 유지. 캐스팅 금지.**
운영 대시보드 유저수는 deep_data 안에서 `COUNT(DISTINCT user_id)`로 세면 되고, votes 조인이 꼭 필요한 특수 분석에서만 그 시점에 캐스팅한다.
(단 hash·numeric이 동일인의 다른 id일 수 있어 distinct 카운트가 약간 이중계상될 수 있음 → 운영 추세엔 무방, 한계로 기록)

In [ ]:
%%bigquery --project your-gcp-project
-- user_id 형식 월별 추이 (숫자 변환 생존율)
SELECT
  FORMAT_DATE('%Y-%m', DATE(local_timestamp)) AS ym,
  COUNT(*)                                          AS row_cnt,
  COUNTIF(user_id IS NOT NULL AND user_id!='')      AS has_user_raw,
  COUNTIF(SAFE_CAST(user_id AS INT64) IS NOT NULL)  AS cast_survive
FROM `your-gcp-project.log_Data.deep_data`
WHERE event_key = 'complete_question_question'
GROUP BY ym ORDER BY ym;

In [ ]:
%%bigquery --project your-gcp-project
-- user_id 형식 분류 (hash / numeric / empty)
WITH classified AS (
  SELECT
    CASE
      WHEN user_id IS NULL OR user_id = ''       THEN 'empty'
      WHEN REGEXP_CONTAINS(user_id, r'^[0-9]+$') THEN 'numeric'
      ELSE 'hash'
    END AS id_type,
    user_id
  FROM `your-gcp-project.log_Data.deep_data`
  WHERE event_key = 'complete_question_question'
)
SELECT id_type, COUNT(*) AS row_cnt, COUNT(DISTINCT user_id) AS distinct_ids
FROM classified
GROUP BY id_type ORDER BY row_cnt DESC;

## 4. `int_ping_events` — PING 발신 정의 레이어 (최종)

**PING 정의를 담는 단일 관리 지점.** 정의가 바뀌면 이 뷰 하나만 수정하고, 상위 마트는 그대로 둔다.
(SAFE_CAST 제거, empty 제외 버전)

In [ ]:
%%bigquery --project your-gcp-project
-- PING 발신 정의 단일 관리 지점. 정의 바뀌면 이 뷰만 수정.
CREATE OR REPLACE VIEW `your-gcp-project.log_Data.int_ping_events` AS
SELECT
  id                                             AS event_id,        -- 이벤트 PK (중복 없음 확인됨)
  user_id                                        AS sender_user_id,  -- STRING 원본 유지 (hash/numeric 혼재, 캐스팅 금지)
  DATETIME(local_timestamp, 'Asia/Seoul')        AS event_datetime_kst,
  DATE(DATETIME(local_timestamp, 'Asia/Seoul'))  AS event_date_kst
FROM `your-gcp-project.log_Data.deep_data`
WHERE event_key = 'complete_question_question'
  AND user_id IS NOT NULL AND user_id != '';     -- empty 제외 (유저수 집계 대상만)
-- 정의: complete_question_question 1건 = PING 발신 1건 (하루 여러 건 = 여러 발신)
-- 변경 예시: 유저별 하루 1회로 보려면 마트에서 COUNT(DISTINCT sender_user_id) 사용
--            다른 이벤트로 바꾸려면 event_key 조건만 교체

In [ ]:
%%bigquery --project your-gcp-project
-- 검증: 일별 발신 유저/건수 (전 구간, ping_users가 0이 아니어야 정상)
SELECT
  event_date_kst,
  COUNT(*)                        AS ping_count,
  COUNT(DISTINCT sender_user_id)  AS ping_users
FROM `your-gcp-project.log_Data.int_ping_events`
GROUP BY event_date_kst
ORDER BY event_date_kst;

## 5. 다음 단계

1. **`int_hint_opens`** — 힌트 열람 정의 레이어 (팀 논의 #2·#3 결정 후: deep_data `click_question_hint` 기준으로 갈지, votes `opened_times` 기준으로 갈지)
2. **`mart_ops_daily`** — 일별 통합 마트 (1 row = 1 date). DAU/PING/힌트(deep_data) + 결제/탈퇴/가입(votes)을 날짜로 조인, 전일·전주 비교와 이상 플래그 포함
3. **`mart_retention_cohort`** — (팀 논의 #5 결정 시) votes 분석 리텐션을 코호트 grain으로 별도 관리

> 지표별 소스가 갈리므로(deep_data=행동, votes=트랜잭션), 각 소스를 **동일 날짜 grain으로 먼저 집계한 뒤 date로 조인**한다. fact끼리 raw 조인 금지.

In [ ]:
%%bigquery --project your-gcp-project
-- votes 소스 실제 날짜 범위 (deep_data 축 2023-06~2024-04와 겹치는지 확정)
SELECT 'signup'  AS src, MIN(DATE(signup_at))    AS min_d, MAX(DATE(signup_at))    AS max_d, COUNT(*) AS row_cnt
FROM `your-gcp-project.stage.stg_users` WHERE is_valid_user AND NOT is_staff_account
UNION ALL SELECT 'payment', MIN(DATE(purchased_at)), MAX(DATE(purchased_at)), COUNT(*) FROM `your-gcp-project.stage.stg_payment`
UNION ALL SELECT 'withdraw', MIN(DATE(withdrawn_at)), MAX(DATE(withdrawn_at)), COUNT(*) FROM `your-gcp-project.stage.stg_withdraw`;

In [ ]:
%%bigquery --project your-gcp-project
-- (1) 두 이벤트 규모/기간/property 비교
SELECT
  event_key,
  COUNT(*)                        AS row_cnt,
  COUNT(DISTINCT user_id)         AS users,
  MIN(DATE(local_timestamp))      AS first_d,
  MAX(DATE(local_timestamp))      AS last_d,
  COUNT(DISTINCT event_properties) AS distinct_props   -- property 구조가 다른지
FROM `your-gcp-project.log_Data.deep_data`
WHERE event_key IN ('click_question_hint','click_question_hint_initial')
GROUP BY event_key;

In [ ]:
%%bigquery --project your-gcp-project
-- (2) 유저 겹침: 둘 다 한 사람 vs 한쪽만 (포함관계·단계 판별)
WITH u AS (
  SELECT DISTINCT
    user_id,
    event_key
  FROM `your-gcp-project.log_Data.deep_data`
  WHERE event_key IN ('click_question_hint','click_question_hint_initial')
    AND user_id IS NOT NULL AND user_id != ''
)
SELECT
  COUNTIF(has_hint AND has_initial)      AS both,
  COUNTIF(has_hint AND NOT has_initial)  AS hint_only,
  COUNTIF(NOT has_hint AND has_initial)  AS initial_only
FROM (
  SELECT user_id,
    LOGICAL_OR(event_key='click_question_hint')         AS has_hint,
    LOGICAL_OR(event_key='click_question_hint_initial') AS has_initial
  FROM u GROUP BY user_id
);

In [ ]:
%%bigquery --project your-gcp-project
-- 유저-일 단위 발신 건수 분포 (헤비 임계값 N 정하기)
WITH user_day AS (
  SELECT event_date_kst, sender_user_id, COUNT(*) AS sends
  FROM `your-gcp-project.log_Data.int_ping_events`
  GROUP BY 1,2
)
SELECT
  ROUND(AVG(sends),2)  AS avg_sends_per_user_day,
  APPROX_QUANTILES(sends, 100)[OFFSET(50)] AS p50,
  APPROX_QUANTILES(sends, 100)[OFFSET(75)] AS p75,
  APPROX_QUANTILES(sends, 100)[OFFSET(90)] AS p90,
  MAX(sends) AS max_sends
FROM user_day;

In [ ]:
%%bigquery --project your-gcp-project
-- (1) question 관련 event_key 전부 + 규모 (세트 완료 이벤트 존재 여부)
SELECT event_key, COUNT(*) AS row_cnt, COUNT(DISTINCT user_id) AS users
FROM `your-gcp-project.log_Data.deep_data`
WHERE event_key LIKE '%question%'
GROUP BY event_key
ORDER BY row_cnt DESC;

In [ ]:
%%bigquery --project your-gcp-project
-- (2) event_properties 실제 값 까보기 (세트/문항 식별자, 지목대상 있나)
SELECT event_properties
FROM `your-gcp-project.log_Data.deep_data`
WHERE event_key = 'complete_question_question'
  AND user_id IS NOT NULL AND user_id != ''
LIMIT 5;

In [ ]:
%%bigquery --project your-gcp-project
-- 결제 일별 연속성 체크 (0으로 끊기는 구간 있나)
SELECT
  DATE(purchased_at) AS date_kst,
  COUNT(DISTINCT user_id) AS paying_users,
  COUNT(*)                AS payment_count,
  SUM(heart_amount)       AS hearts_sold
FROM `your-gcp-project.stage.stg_payment`
WHERE DATE(purchased_at) BETWEEN '2023-06-01' AND '2024-04-30'
GROUP BY date_kst
ORDER BY date_kst;

## Mart 생성

### 1. mart_ops_daily

In [ ]:
%%bigquery --project your-gcp-project
-- PING 발신 정의 단일 관리 지점. 정의 바뀌면 이 뷰만 수정.
CREATE OR REPLACE VIEW `your-gcp-project.log_Data.int_ping_events` AS
SELECT
  id                                             AS event_id,        -- 이벤트 PK (중복 없음 확인됨)
  user_id                                        AS sender_user_id,  -- STRING 원본 유지 (hash/numeric 혼재, 캐스팅 금지)
  DATETIME(local_timestamp, 'Asia/Seoul')        AS event_datetime_kst,
  DATE(DATETIME(local_timestamp, 'Asia/Seoul'))  AS event_date_kst
FROM `your-gcp-project.log_Data.deep_data`
WHERE event_key = 'complete_question_question'      -- 질문 답하며 지목 = 발신
  AND user_id IS NOT NULL AND user_id != '';        -- empty 제외 (유저수 집계 대상만)
-- 정의: complete_question_question 1건 = PING 발신 1건 (하루 여러 건 = 여러 발신)
-- 변경 시: event_key 조건만 교체하면 됨

In [ ]:
%%bigquery --project your-gcp-project
-- 힌트 열람 정의 단일 관리 지점 (click_question_hint = 힌트 열람 1건)
CREATE OR REPLACE VIEW `your-gcp-project.log_Data.int_hint_opens` AS
SELECT
  id                                             AS event_id,
  user_id                                        AS user_id,          -- STRING 원본 (캐스팅 금지)
  DATETIME(local_timestamp, 'Asia/Seoul')        AS event_datetime_kst,
  DATE(DATETIME(local_timestamp, 'Asia/Seoul'))  AS event_date_kst
FROM `your-gcp-project.log_Data.deep_data`
WHERE event_key = 'click_question_hint'             -- initial은 이 집합의 부분집합이라 제외(이중계상 방지)
  AND user_id IS NOT NULL AND user_id != '';
-- 데이터 시작: 2023-09-13 (그 전엔 이벤트 없음 → 힌트 라인은 9월부터)

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE TABLE `your-gcp-project.mart.mart_ops_daily`
PARTITION BY date_kst
OPTIONS(description='운영 모니터링 일별 통합 마트. 1 row = 1 KST date. 시작 2023-06-21. deep_data 공백(7/07~8/10)은 NULL') AS
WITH
bounds AS (
  SELECT DATE '2023-04-01' AS min_d, DATE '2024-04-30' AS max_d   -- deep_data 실제 시작 6/21
),
calendar AS (
  SELECT d AS date_kst
  FROM bounds, UNNEST(GENERATE_DATE_ARRAY(min_d, max_d)) AS d
),
dau AS (
  SELECT DATE(DATETIME(local_timestamp,'Asia/Seoul')) AS date_kst,
         COUNT(DISTINCT user_id) AS dau
  FROM `your-gcp-project.log_Data.deep_data`
  WHERE event_key='launch_app' AND user_id IS NOT NULL AND user_id!=''
  GROUP BY 1
),
ping AS (
  SELECT event_date_kst AS date_kst,
         COUNT(DISTINCT sender_user_id) AS ping_users,
         COUNT(*)                       AS ping_count
  FROM `your-gcp-project.log_Data.int_ping_events` GROUP BY 1
),
hint AS (
  SELECT event_date_kst AS date_kst,
         COUNT(DISTINCT user_id) AS hint_users,
         COUNT(*)                AS hint_count
  FROM `your-gcp-project.log_Data.int_hint_opens` GROUP BY 1
),
signup AS (
  SELECT DATE(signup_at) AS date_kst, COUNT(*) AS new_users
  FROM `your-gcp-project.stage.stg_users`
  WHERE is_valid_user AND NOT is_staff_account GROUP BY 1
),
payment AS (
  SELECT DATE(purchased_at) AS date_kst,
         COUNT(DISTINCT user_id) AS paying_users,
         COUNT(*)                AS payment_count,
         SUM(heart_amount)       AS hearts_sold
  FROM `your-gcp-project.stage.stg_payment` GROUP BY 1
),
churn AS (
  SELECT DATE(withdrawn_at) AS date_kst, COUNT(*) AS churn_users
  FROM `your-gcp-project.stage.stg_withdraw`
  WHERE is_valid_withdraw GROUP BY 1
),
joined AS (
  SELECT c.date_kst,
    dau.dau               AS dau,           -- deep: NULL 유지(공백=선 끊김)
    ping.ping_users       AS ping_users,    -- deep
    ping.ping_count       AS ping_count,    -- deep
    hint.hint_users       AS hint_users,    -- deep
    hint.hint_count       AS hint_count,    -- deep
    COALESCE(signup.new_users,0)      AS new_users,      -- votes: 0
    COALESCE(payment.paying_users,0)  AS paying_users,   -- votes: 0
    COALESCE(payment.payment_count,0) AS payment_count,  -- votes: 0
    COALESCE(payment.hearts_sold,0)   AS hearts_sold,    -- votes: 0
    COALESCE(churn.churn_users,0)     AS churn_users     -- votes: 0
  FROM calendar c
  LEFT JOIN dau     USING(date_kst)
  LEFT JOIN ping    USING(date_kst)
  LEFT JOIN hint    USING(date_kst)
  LEFT JOIN signup  USING(date_kst)
  LEFT JOIN payment USING(date_kst)
  LEFT JOIN churn   USING(date_kst)
),
enriched AS (
  SELECT *,
    SAFE_DIVIDE(ping_count, ping_users)            AS sends_per_user,
    SAFE_DIVIDE(hint_users, NULLIF(ping_users,0))  AS hint_rate,
    LAG(dau)            OVER w AS dau_prev_day,
    LAG(dau,7)          OVER w AS dau_prev_week,
    LAG(new_users,7)    OVER w AS new_users_prev_week,
    LAG(ping_users,7)   OVER w AS ping_users_prev_week,
    LAG(paying_users,7) OVER w AS paying_users_prev_week,
    AVG(dau) OVER (ORDER BY date_kst ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS dau_ma7
  FROM joined
  WINDOW w AS (ORDER BY date_kst)
)
SELECT *,
  SAFE_DIVIDE(dau - dau_prev_week,  NULLIF(dau_prev_week,0))                           AS dau_wow_rate,
  SAFE_DIVIDE(new_users - new_users_prev_week, NULLIF(new_users_prev_week,0))          AS new_users_wow_rate,
  SAFE_DIVIDE(ping_users - ping_users_prev_week, NULLIF(ping_users_prev_week,0))       AS ping_users_wow_rate,
  SAFE_DIVIDE(paying_users - paying_users_prev_week, NULLIF(paying_users_prev_week,0)) AS paying_users_wow_rate,
  ( SAFE_DIVIDE(dau - dau_prev_week, NULLIF(dau_prev_week,0)) <= -0.20
    OR SAFE_DIVIDE(dau - dau_ma7,    NULLIF(dau_ma7,0))       <= -0.30 ) AS dau_alert_drop
FROM enriched;

### 2. mart_retention_cohort (retention 단독, 보조 지표)

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE TABLE `your-gcp-project.mart.mart_retention_cohort`
PARTITION BY cohort_date
OPTIONS(description='코호트별 D+n 재발신 리텐션. 운영 지표 아님(분석 참고). 소스: mart_activation_retention') AS
SELECT
  signup_date                                          AS cohort_date,
  COUNT(*)                                             AS activated_users,
  COUNTIF(retained_within_d1)                          AS d1_retained,
  COUNTIF(retained_within_d3)                          AS d3_retained,
  COUNTIF(retained_within_d7)                          AS d7_retained,
  COUNTIF(retained_within_d10)                         AS d10_retained,
  SAFE_DIVIDE(COUNTIF(retained_within_d1),  COUNT(*))  AS d1_retention_rate,
  SAFE_DIVIDE(COUNTIF(retained_within_d3),  COUNT(*))  AS d3_retention_rate,
  SAFE_DIVIDE(COUNTIF(retained_within_d7),  COUNT(*))  AS d7_retention_rate,
  SAFE_DIVIDE(COUNTIF(retained_within_d10), COUNT(*))  AS d10_retention_rate
FROM `your-gcp-project.mart.mart_activation_retention`
GROUP BY cohort_date;

### 생성 후 검증

In [ ]:
%%bigquery --project your-gcp-project
-- 4~5월이 어떻게 채워지는지 (votes는 값, deep는 NULL 예상)
SELECT date_kst, dau, ping_users, new_users, paying_users, churn_users
FROM `your-gcp-project.mart.mart_ops_daily`
WHERE date_kst IN ('2023-04-15','2023-05-15','2023-06-01','2023-06-25','2023-09-19')
ORDER BY date_kst;

In [ ]:
%%bigquery --project your-gcp-project
SELECT date_kst, dau, ping_users, new_users, paying_users
FROM `your-gcp-project.mart.mart_ops_daily`
WHERE date_kst BETWEEN '2023-07-05' AND '2023-07-10'
ORDER BY date_kst;

In [ ]:
%%bigquery --project your-gcp-project
-- 행 수, 날짜 범위, 각 지표 NULL/합계, 공백 구간 확인
SELECT
  COUNT(*)                                   AS total_days,        -- 315일이어야(6/21~4/30)
  MIN(date_kst)                              AS min_d,
  MAX(date_kst)                              AS max_d,
  COUNTIF(dau IS NULL)                       AS dau_null_days,     -- deep 공백 = 35일 정도 예상
  COUNTIF(new_users IS NULL)                 AS newuser_null_days, -- 0이어야(votes는 항상 값)
  SUM(new_users)                             AS total_new_users,   -- 가입 총합
  SUM(payment_count)                         AS total_payments,    -- 결제 총합
  ROUND(AVG(sends_per_user),3)               AS avg_intensity      -- 1.3~1.5 예상
FROM `your-gcp-project.mart.mart_ops_daily`;

In [ ]:
%%bigquery --project your-gcp-project
-- date_kst가 진짜 유일한지 (1 row = 1 date 보장)
SELECT COUNT(*) AS rows, COUNT(DISTINCT date_kst) AS distinct_dates
FROM `your-gcp-project.mart.mart_ops_daily`;
-- 두 값이 같아야 정상. 다르면 grain 깨진 것.

In [ ]:
%%bigquery --project your-gcp-project
-- 특정일(정상 구간) 마트 값 vs 원본 직접 집계 대조
WITH mart AS (
  SELECT dau, ping_users, new_users, paying_users
  FROM `your-gcp-project.mart.mart_ops_daily` WHERE date_kst='2023-09-15'
),
raw_dau AS (
  SELECT COUNT(DISTINCT user_id) AS dau FROM `your-gcp-project.log_Data.deep_data`
  WHERE event_key='launch_app' AND DATE(DATETIME(local_timestamp,'Asia/Seoul'))='2023-09-15'
    AND user_id IS NOT NULL AND user_id!=''
),
raw_signup AS (
  SELECT COUNT(*) AS new_users FROM `your-gcp-project.stage.stg_users`
  WHERE DATE(signup_at)='2023-09-15' AND is_valid_user AND NOT is_staff_account
)
SELECT m.dau AS mart_dau, r.dau AS raw_dau,
       m.new_users AS mart_signup, s.new_users AS raw_signup
FROM mart m, raw_dau r, raw_signup s;
-- mart_dau=raw_dau, mart_signup=raw_signup 이면 조인 정상

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*)                                       AS cohorts,
  MIN(cohort_date)                               AS min_c,
  MAX(cohort_date)                               AS max_c,
  COUNTIF(d7_retention_rate NOT BETWEEN 0 AND 1) AS bad_rate,   -- 0이어야(비율은 0~1)
  ROUND(AVG(d7_retention_rate)*100,1)            AS avg_d7_pct
FROM `your-gcp-project.mart.mart_retention_cohort`;
-- bad_rate=0, avg_d7_pct가 votes 분석 때 본 값과 비슷하면 정상

## etc

In [ ]:
%%bigquery --project your-gcp-project
-- deep_data가 6월에 '일별로' 실제 있는지 직접 확인 (마트 거치지 않고)
SELECT
  DATE(DATETIME(local_timestamp,'Asia/Seoul')) AS d,
  COUNTIF(event_key='launch_app')                    AS launch_rows,
  COUNT(DISTINCT IF(event_key='launch_app', user_id, NULL))                  AS dau,
  COUNTIF(event_key='complete_question_question')    AS ping_rows,
  COUNT(DISTINCT IF(event_key='complete_question_question', user_id, NULL))  AS ping_users
FROM `your-gcp-project.log_Data.deep_data`
WHERE DATE(DATETIME(local_timestamp,'Asia/Seoul')) BETWEEN '2023-06-01' AND '2023-06-30'
  AND user_id IS NOT NULL AND user_id!=''
GROUP BY d ORDER BY d;

In [ ]:
%%bigquery --project your-gcp-project
-- deep_data 7월 일별 실제 존재 여부 (launch_app 기준)
SELECT
  DATE(DATETIME(local_timestamp,'Asia/Seoul')) AS d,
  COUNT(*) AS row_cnt,
  COUNT(DISTINCT user_id) AS users
FROM `your-gcp-project.log_Data.deep_data`
WHERE event_key='launch_app'
  AND DATE(DATETIME(local_timestamp,'Asia/Seoul')) BETWEEN '2023-07-01' AND '2023-08-15'
GROUP BY d ORDER BY d;

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*)                AS total_rows,          -- 월별 row_cnt 합과 같아야 정상
  COUNT(DISTINCT user_id) AS total_distinct_users -- 월별 users 합보다 작아야 정상(중복 제거)
FROM `your-gcp-project.log_Data.deep_data`;

In [ ]:
%%bigquery --project your-gcp-project
SELECT COUNT(*) AS signup_in_window
FROM `your-gcp-project.stage.stg_users`
WHERE is_valid_user AND NOT is_staff_account
  AND DATE(signup_at) BETWEEN '2023-06-21' AND '2024-04-30';

In [ ]:
%%bigquery --project your-gcp-project
-- DAU 기준 가장 활발한 시기 찾기 (모든 핵심지표 값 있는 구간)
SELECT
  date_kst, dau, ping_users, hint_users, paying_users, churn_users
FROM `your-gcp-project.mart.mart_ops_daily`
WHERE dau IS NOT NULL          -- deep 공백 제외
  AND hint_users > 0           -- 힌트도 나오는 구간(9/13 이후)만
ORDER BY dau DESC
LIMIT 20;

In [ ]:
%%bigquery --project your-gcp-project
-- 최근 날짜(4/30) 데일리 보고서용 숫자
SELECT
  date_kst,
  dau, ping_users, hint_users, new_users, paying_users, payment_count, hearts_sold, churn_users,
  ROUND(dau_wow_rate*100,1)          AS dau_wow_pct,
  ROUND(ping_users_wow_rate*100,1)   AS ping_wow_pct,
  ROUND(paying_users_wow_rate*100,1) AS pay_wow_pct,
  dau_alert_drop
FROM `your-gcp-project.mart.mart_ops_daily`
WHERE date_kst IN ('2024-04-30','2024-04-29','2024-04-23')  -- 당일/전일/전주
ORDER BY date_kst;

In [ ]:
%%bigquery --project your-gcp-project
-- 9/19 데일리 보고서용 숫자 (당일 + 전일 + 전주 + 증감)
SELECT
  date_kst,
  dau, ping_users, hint_users, new_users, paying_users, payment_count, hearts_sold, churn_users,
  ROUND(dau_wow_rate*100,1)          AS dau_wow_pct,
  ROUND(new_users_wow_rate*100,1)    AS newuser_wow_pct,
  ROUND(ping_users_wow_rate*100,1)   AS ping_wow_pct,
  ROUND(paying_users_wow_rate*100,1) AS pay_wow_pct,
  dau_alert_drop
FROM `your-gcp-project.mart.mart_ops_daily`
WHERE date_kst IN ('2023-09-18','2023-09-19','2023-09-12')  -- 당일/전일/전주
ORDER BY date_kst;